In [58]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

import os

api_key = os.getenv("OPENROUTER_API_KEY")



In [59]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [60]:

llm = ChatOpenAI(
      model="openrouter/free",
       base_url="https://openrouter.ai" + "/api/v1",
       api_key=api_key,
       temperature=0.2,
       max_tokens=4000,
       max_retries=3,
)

In [61]:
def chat_node(state:ChatState):
    message = state['messages']
    response = llm.invoke(message)
    return {'messages':[response]}


In [62]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)
# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer=checkpointer);

In [63]:
# thread_id = '1'
# config = {'configurable': {'thread_id':thread_id}}
# initial_state = {
# 'messages': [HumanMessage(content='What is the capital of india')] }

# chatbot.invoke(initial_state,config=config)['messages'][1].content

In [64]:
thread_id = '1'

while True:
    user_message = input('Type here: ')
    display(user_message)
    if user_message.strip().lower() in ['exit','quit','bye']:
        break
        # chatbot.invoke('')
    config = {'configurable': {'thread_id':thread_id}}
    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
    display('AI',response['messages'][-1].content)
    # display('AI',response['messages'][0].content)
    # display('AI',response['messages'][1].content)




'what is my name'

'AI'

"\n\nI don't have access to your name or any personal information about you. If you'd like to tell me your name, I'd be happy to use it in our conversation!"

'quit'